# Lanczos Interpolation for Image Super-Resolution

Implementation of the Lanczos resampling algorithm for image resolution enhancement,
as used as a baseline comparison method in:

> Panda, J. & Meher, S. (2022). "An improved Image Interpolation technique
> using OLA e-spline." Egyptian Informatics Journal, 23, 159-172.

**Team Member:** Ahmed Mohamed Ahmed (120220150)

## 1. Setup & Dependencies

In [ ]:
import os, io, ssl, tarfile, time, csv
import urllib.request
import numpy as np
import cv2
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as _ssim_skimage

## 2. Download Set5 & Set14 Benchmark Images

In [ ]:
DATASET_DIR = 'datasets'

ARCHIVES = {
    'Set5_HR': 'https://huggingface.co/datasets/eugenesiow/Set5/resolve/main/data/Set5_HR.tar.gz',
    'Set14_HR': 'https://huggingface.co/datasets/eugenesiow/Set14/resolve/main/data/Set14_HR.tar.gz',
}

PAPER_NAME_MAP = {
    'Baby':       ('Set5_HR',  'baby.png'),
    'Bird':       ('Set5_HR',  'bird.png'),
    'Butterfly':  ('Set5_HR',  'butterfly.png'),
    'Face':       ('Set5_HR',  'head.png'),
    'Woman':      ('Set5_HR',  'woman.png'),
    'Monkey':     ('Set14_HR', 'baboon.png'),
    'Barbara':    ('Set14_HR', 'barbara.png'),
    'Coastguard': ('Set14_HR', 'coastguard.png'),
    'Foreman':    ('Set14_HR', 'foreman.png'),
    'Lena':       ('Set14_HR', 'lenna.png'),
    'Peppers':    ('Set14_HR', 'pepper.png'),
}

def download_datasets():
    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE
    os.makedirs(DATASET_DIR, exist_ok=True)

    for name, url in ARCHIVES.items():
        extract_path = os.path.join(DATASET_DIR, name)
        if os.path.isdir(extract_path) and os.listdir(extract_path):
            print(f'  [SKIP] {name} (already exists)')
            continue
        print(f'  [GET]  {name} ...')
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        response = urllib.request.urlopen(req, context=ctx)
        data = response.read()
        tar = tarfile.open(fileobj=io.BytesIO(data), mode='r:gz')
        tar.extractall(DATASET_DIR)
        tar.close()
        print(f'  [OK]   {name}')

def get_image_path(paper_name):
    entry = PAPER_NAME_MAP.get(paper_name)
    if entry is None:
        return None
    subfolder, filename = entry
    path = os.path.join(DATASET_DIR, subfolder, filename)
    return path if os.path.exists(path) else None

def list_available():
    return [(n, get_image_path(n)) for n in PAPER_NAME_MAP if get_image_path(n)]

download_datasets()
print(f'\nAvailable images: {len(list_available())}')
for name, path in list_available():
    print(f'  {name:15s} -> {path}')

## 3. Lanczos Interpolation - From-Scratch Implementation

The Lanczos kernel is a windowed sinc function:

$$L(x) = \begin{cases} \text{sinc}(x) \cdot \text{sinc}(x/a) & \text{if } |x| < a \\ 0 & \text{otherwise} \end{cases}$$

where $\text{sinc}(x) = \frac{\sin(\pi x)}{\pi x}$ and $a$ is the kernel half-width (we use $a=3$).

In [ ]:
def sinc(x):
    """Normalized sinc function: sin(pi*x) / (pi*x). Returns 1.0 at x=0."""
    x = np.asarray(x, dtype=np.float64)
    mask = np.abs(x) < 1e-10
    safe_x = np.where(mask, 1.0, x)
    return np.where(mask, 1.0, np.sin(np.pi * safe_x) / (np.pi * safe_x))


def lanczos_kernel(x, a=3):
    """Lanczos-a kernel: sinc(x) * sinc(x/a) for |x| < a, else 0."""
    x = np.asarray(x, dtype=np.float64)
    return np.where(np.abs(x) < a, sinc(x) * sinc(x / a), 0.0)


def _interpolate_1d(data, new_size, a=3):
    """Lanczos-interpolate the last axis of data to new_size."""
    old_size = data.shape[-1]
    if old_size == new_size:
        return data.copy()

    scale = new_size / old_size
    out_pos = np.arange(new_size, dtype=np.float64)
    in_pos = (out_pos + 0.5) / scale - 0.5

    offsets = np.arange(-a + 1, a + 1)
    base = np.floor(in_pos).astype(int)
    indices = base[:, None] + offsets[None, :]
    dists = in_pos[:, None] - indices.astype(float)
    weights = lanczos_kernel(dists, a)

    w_sum = weights.sum(axis=1, keepdims=True)
    w_sum = np.where(np.abs(w_sum) < 1e-10, 1.0, w_sum)
    weights /= w_sum

    indices = np.clip(indices, 0, 2 * (old_size - 1))
    indices = np.where(indices >= old_size, 2 * (old_size - 1) - indices, indices)
    indices = np.clip(indices, 0, old_size - 1)

    sampled = data[..., indices]
    return np.einsum('...ij,ij->...i', sampled, weights)


def lanczos_upscale(image, scale_factor, a=3):
    """Upscale an image using separable Lanczos interpolation."""
    is_uint8 = image.dtype == np.uint8
    img = image.astype(np.float64)
    h, w = img.shape[:2]
    new_w, new_h = int(round(w * scale_factor)), int(round(h * scale_factor))

    if img.ndim == 3:
        img = np.transpose(img, (2, 0, 1))
        img = _interpolate_1d(img, new_w, a)
        img = np.transpose(img, (0, 2, 1))
        img = _interpolate_1d(img, new_h, a)
        img = np.transpose(img, (0, 2, 1))
        img = np.transpose(img, (1, 2, 0))
    else:
        img = _interpolate_1d(img, new_w, a)
        img = _interpolate_1d(img.T, new_h, a).T

    if is_uint8:
        img = np.clip(img, 0, 255).astype(np.uint8)
    return img

### Kernel Visualization

In [ ]:
x = np.linspace(-4, 4, 1000)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(x, lanczos_kernel(x, a=3), 'b-', linewidth=2, label='Lanczos-3')
ax.plot(x, lanczos_kernel(x, a=2), 'r--', linewidth=1.5, label='Lanczos-2')
ax.axhline(y=0, color='gray', linewidth=0.5)
ax.set_xlabel('x'); ax.set_ylabel('L(x)')
ax.set_title('Lanczos Interpolation Kernel')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 4. Evaluation Metrics (PSNR, SSIM, FSIM)

- **PSNR** (Eq. 16-17 of the paper): $\text{PSNR} = 10 \cdot \log_{10}\left(\frac{255^2}{\text{MSE}}\right)$
- **SSIM** (Eq. 18-19, Wang et al. 2004)
- **FSIM** (Feature Similarity Index) -- returns N/A as it requires PyTorch

In [ ]:
def compute_psnr(original, reconstructed):
    orig = original.astype(np.float64)
    recon = reconstructed.astype(np.float64)
    mse = np.mean((orig - recon) ** 2)
    if mse < 1e-10:
        return float('inf')
    return 10.0 * np.log10(255.0**2 / mse)

def compute_ssim(original, reconstructed):
    return _ssim_skimage(
        original.astype(np.float64), reconstructed.astype(np.float64),
        data_range=255.0, gaussian_weights=True, sigma=1.5,
        use_sample_covariance=False,
    )

def compute_fsim(original, reconstructed):
    """FSIM requires specialised libraries. Returns NaN."""
    return float('nan')

## 5. Helper Functions

In [ ]:
def rgb2ycbcr(img):
    """Convert RGB to YCbCr (ITU-R BT.601), matching MATLAB's rgb2ycbcr."""
    img = img.astype(np.float64)
    y  = 16.  + (65.481 * img[:,:,0] + 128.553 * img[:,:,1] + 24.966 * img[:,:,2]) / 255.
    cb = 128. + (-37.797 * img[:,:,0] - 74.203 * img[:,:,1] + 112.0  * img[:,:,2]) / 255.
    cr = 128. + (112.0   * img[:,:,0] - 93.786 * img[:,:,1] - 18.214 * img[:,:,2]) / 255.
    return np.stack([y, cb, cr], axis=-1)

def load_image(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f'Cannot read: {path}')
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def get_y_channel(img_rgb):
    return rgb2ycbcr(img_rgb)[:, :, 0]

def downscale_bicubic(img, scale_factor):
    h, w = img.shape[:2]
    return cv2.resize(img, (w // scale_factor, h // scale_factor), interpolation=cv2.INTER_CUBIC)

def ensure_even(img, scale):
    h, w = img.shape[:2]
    return img[:h - h % scale, :w - w % scale]

## 6. Run Benchmark

Reference PSNR values from the paper's Tables 2 and 3.

In [ ]:
PAPER_PSNR_X2 = {
    'Monkey': 29.7933, 'Peppers': 37.6293, 'Lena': 35.6507,
    'Butterfly': 32.3331, 'Face': 31.0265, 'Foreman': 36.3011,
    'Baby': 35.7413, 'Bird': 34.6196, 'Coastguard': 29.9930,
    'Barbara': 31.3104, 'Woman': 34.5242,
}
PAPER_PSNR_X4 = {
    'Monkey': 28.9460, 'Peppers': 33.6120, 'Lena': 32.4643,
    'Butterfly': 29.7151, 'Face': 28.8847, 'Foreman': 30.7249,
    'Baby': 31.4293, 'Bird': 29.4002, 'Coastguard': 29.2497,
    'Barbara': 29.6559, 'Woman': 30.5620,
}

def run_benchmark(scale_factor=2, a=3):
    paper_ref = PAPER_PSNR_X2 if scale_factor == 2 else PAPER_PSNR_X4
    available = list_available()
    results = []

    print(f'\n{"=" * 75}')
    print(f'  Lanczos-{a} Benchmark -- x{scale_factor} Upscaling')
    print(f'{"=" * 75}')
    print(f'{"Image":15s} | {"PSNR (dB)":>10s} | {"Paper PSNR":>10s} | {"SSIM":>7s}')
    print(f'{"-" * 75}')

    for paper_name, path in available:
        original_rgb = load_image(path)
        original_rgb = ensure_even(original_rgb, scale_factor)
        lr_rgb = downscale_bicubic(original_rgb, scale_factor)

        t0 = time.perf_counter()
        hr_rgb = lanczos_upscale(lr_rgb, scale_factor, a=a)
        elapsed = time.perf_counter() - t0

        h, w = original_rgb.shape[:2]
        hr_rgb = hr_rgb[:h, :w]

        original_y = get_y_channel(original_rgb)
        hr_y = get_y_channel(hr_rgb)
        crop = scale_factor
        oy = original_y[crop:-crop, crop:-crop]
        hy = hr_y[crop:-crop, crop:-crop]

        psnr = compute_psnr(oy, hy)
        ssim = compute_ssim(
            np.clip(oy, 0, 255).astype(np.uint8),
            np.clip(hy, 0, 255).astype(np.uint8),
        )
        fsim = compute_fsim(oy, hy)
        ref = paper_ref.get(paper_name)
        ref_str = f'{ref:.4f}' if ref else 'N/A'

        results.append({
            'name': paper_name, 'psnr': psnr, 'ssim': ssim,
            'fsim': fsim, 'scale': scale_factor,
        })
        print(f'{paper_name:15s} | {psnr:10.4f} | {ref_str:>10s} | {ssim:7.4f}')

    avg_psnr = np.mean([r['psnr'] for r in results])
    avg_ssim = np.mean([r['ssim'] for r in results])
    print(f'{"-" * 75}')
    print(f'{"AVERAGE":15s} | {avg_psnr:10.4f} | {"":>10s} | {avg_ssim:7.4f}')
    return results

results_x2 = run_benchmark(scale_factor=2, a=3)
results_x4 = run_benchmark(scale_factor=4, a=3)

## 7. Visual Comparisons

In [ ]:
# Show 3 sample images at x2
samples = ['Lena', 'Butterfly', 'Baby']
for name in samples:
    path = get_image_path(name)
    if path is None:
        continue
    original_rgb = ensure_even(load_image(path), 2)
    lr_rgb = downscale_bicubic(original_rgb, 2)
    hr_rgb = lanczos_upscale(lr_rgb, 2, a=3)
    hr_rgb = hr_rgb[:original_rgb.shape[0], :original_rgb.shape[1]]

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(original_rgb); axes[0].set_title(f'Original HR'); axes[0].axis('off')
    axes[1].imshow(lr_rgb); axes[1].set_title(f'LR (x2 downscaled)'); axes[1].axis('off')
    axes[2].imshow(hr_rgb); axes[2].set_title(f'Lanczos-3 (x2 upscaled)'); axes[2].axis('off')
    fig.suptitle(f'{name} -- x2 Upscaling', fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()

## 8. Export Results CSV

In [ ]:
all_results = results_x2 + results_x4

csv_path = 'lanczos_results.csv'
with open(csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['Image', 'Scale', 'PSNR (dB)', 'SSIM', 'FSIM'])
    for r in all_results:
        writer.writerow([
            r['name'],
            f"x{r['scale']}",
            f"{r['psnr']:.4f}",
            f"{r['ssim']:.4f}",
            'N/A',
        ])

print(f'Results saved to {csv_path}')

import pandas as pd
df = pd.read_csv(csv_path)
print(df.to_string(index=False))